# Phase 1: data structure and loader check

**CRISP-DM stage:** Data understanding

**Runs on:** Kaggle, with the GPU switched off, because no neural network runs here. The private dataset `revathyh/mvtec-ad2-sheet-metal-vial` is attached, so nothing is downloaded from MVTec.

**Questions it answers:** which folders hold the training, validation and test images, where are the masks, and can anomalib read these files?

## 1. Find the data

We search for the folders instead of typing a fixed path. anomalib downloads the dataset if it cannot find `root/category`, and its download link no longer works, so a wrong path must stop the notebook here.

In [ ]:
from pathlib import Path

INPUT_FOLDER = Path("/kaggle/input")
PRODUCT_NAMES = ["sheet_metal", "vial"]

# Maps each product name to its root, the folder that contains the product folder
product_roots = {}

for train_folder in INPUT_FOLDER.rglob("train"):
    product_folder = train_folder.parent
    if product_folder.name in PRODUCT_NAMES:
        product_roots[product_folder.name] = product_folder.parent

for product_name in PRODUCT_NAMES:
    if product_name not in product_roots:
        # Stop here, so anomalib never tries its download
        raise FileNotFoundError("Could not find the folder for " + product_name)
    print(product_name, "root:", product_roots[product_name])

## 2. Folder structure

We count the files in each folder. Printing every file name would mean about 1,700 lines, and the counts answer the question.

In [ ]:
# Collected in a list so the same lines can be printed and saved to a file
report_lines = []

for product_name in PRODUCT_NAMES:
    root = product_roots[product_name]
    report_lines.append("")
    report_lines.append("Folders for " + product_name + " (file count, then folder)")

    # rglob("*") finds everything at any depth below the product folder
    for path in sorted((root / product_name).rglob("*")):
        if path.is_dir():
            file_count = 0
            for item in path.iterdir():
                if item.is_file():
                    file_count = file_count + 1
            # Folders that only hold other folders are skipped to keep the list short
            if file_count > 0:
                report_lines.append(str(file_count) + "  " + str(path.relative_to(root)))

for line in report_lines:
    print(line)

## 3. Install anomalib

The exact version is pinned so that a later release cannot change how the data is read.

In [ ]:
!pip install anomalib==2.7.0

## 4. Test the loader

`MVTecAD2` is anomalib's object for this dataset.

- `root` is the folder containing the product folder, and `category` is the product folder's name.
- `num_workers=0` loads images in the main process. The default is 8 helper processes, which this small test does not need.
- We rely on three defaults: batch sizes of 32, which have no effect because no batches are made here; the public test set; and no augmentation.

`prepare_data()` downloads only if `root/category` is missing. `setup()` builds the five sets of images.

In [ ]:
import platform

import anomalib
import torch
from anomalib.data import MVTecAD2

report_lines.append("")
report_lines.append("python: " + platform.python_version())
report_lines.append("anomalib: " + anomalib.__version__)
report_lines.append("pytorch: " + torch.__version__)
report_lines.append("gpu available: " + str(torch.cuda.is_available()))

datamodules = {}

for product_name in PRODUCT_NAMES:
    datamodule = MVTecAD2(
        root=product_roots[product_name],
        category=product_name,
        num_workers=0,
    )
    datamodule.prepare_data()
    datamodule.setup()
    datamodules[product_name] = datamodule

    report_lines.append("")
    report_lines.append("Images the loader found for " + product_name)
    report_lines.append("train: " + str(len(datamodule.train_data)))
    report_lines.append("validation: " + str(len(datamodule.val_data)))
    report_lines.append("public test: " + str(len(datamodule.test_public_data)))
    report_lines.append("private test: " + str(len(datamodule.test_private_data)))
    report_lines.append("private mixed test: " + str(len(datamodule.test_private_mixed_data)))

for line in report_lines:
    print(line)

## 5. Open one sample

We read one training image, and one defective test image with its mask. No resizing is set up, so the shapes should be the original image sizes. The full checks on every file come later.

In [ ]:
for product_name in PRODUCT_NAMES:
    datamodule = datamodules[product_name]
    report_lines.append("")
    report_lines.append("One sample from " + product_name)

    train_item = datamodule.train_data[0]
    report_lines.append("train image shape: " + str(list(train_item.image.shape)))

    # label_index is 1 for defective images, so this picks the first defective one
    samples = datamodule.test_public_data.samples
    defective_samples = samples[samples["label_index"] == 1]
    first_defective = defective_samples.index[0]

    test_item = datamodule.test_public_data[first_defective]
    report_lines.append("defective image file: " + Path(test_item.image_path).name)
    report_lines.append("defective image shape: " + str(list(test_item.image.shape)))
    report_lines.append("mask file: " + Path(test_item.mask_path).name)
    report_lines.append("mask shape: " + str(list(test_item.gt_mask.shape)))
    # A mask should only say defect or no defect, so it should hold two values at most
    report_lines.append("mask values: " + str(torch.unique(test_item.gt_mask.int()).tolist()))

for line in report_lines:
    print(line)

## 6. Check nothing downloaded, then save

anomalib's download would leave an archive behind, so finding none in the working folder or next to the data shows it never ran. Files saved in the working folder become the notebook's output, so they survive a disconnect.

In [ ]:
downloaded_archives = list(Path.cwd().rglob("*.tar.gz"))

for product_name in PRODUCT_NAMES:
    root = product_roots[product_name]
    for archive in root.glob("*.tar.gz"):
        downloaded_archives.append(archive)

report_lines.append("")
report_lines.append("archives found after loading: " + str(len(downloaded_archives)))

with open("folder_structure.txt", "w") as report_file:
    for line in report_lines:
        report_file.write(line + "\n")

print("archives found after loading:", len(downloaded_archives))
print("Saved folder_structure.txt")

# Integrity checks

With a curated research dataset, cleaning means verifying, not fixing. These checks only read files and report. Nothing is changed, dropped or repaired.

## 7. Read every file once

We read every file once and keep its facts in one table, called `inventory`, so the checks below do not need to open the files again.

- **Pillow (`PIL`)** opens image files. `image.load()` decodes every pixel, which is why it catches a cut-off file. The faster `verify()` does not decode the pixels, so it can miss one.
- **`hashlib`** computes a SHA-256 fingerprint of a file's bytes. Identical files always get the same fingerprint.
- **A mask's raw values** are 0 and 255. anomalib turns them into 0 and 1 when it reads them, as part 5 showed.

In [ ]:
import hashlib

import numpy
from PIL import Image

CHECK_FOLDERS = [
    "train/good",
    "validation/good",
    "test_public/good",
    "test_public/bad",
    "test_public/ground_truth/bad",
    "test_private",
    "test_private_mixed",
]
MASK_FOLDER = "test_public/ground_truth/bad"


def read_file_facts(path, is_mask):
    """Read one PNG file and return its facts as a dictionary."""
    facts = {
        "opened": False,
        "error": "",
        "width": None,
        "height": None,
        "mode": None,
        "mask_values": None,
    }

    # The fingerprint comes from the raw bytes, so it works even for a damaged file
    with open(path, "rb") as file:
        facts["fingerprint"] = hashlib.sha256(file.read()).hexdigest()

    # try/except so that one damaged file is reported and the check moves on
    try:
        with Image.open(path) as image:
            image.load()
            facts["opened"] = True
            facts["width"] = image.width
            facts["height"] = image.height
            facts["mode"] = image.mode
            if is_mask:
                facts["mask_values"] = numpy.unique(numpy.array(image)).tolist()
    except Exception as error:
        facts["error"] = str(error)

    return facts


inventory = []

for product_name in PRODUCT_NAMES:
    product_folder = product_roots[product_name] / product_name
    for folder_name in CHECK_FOLDERS:
        for path in sorted((product_folder / folder_name).glob("*.png")):
            facts = read_file_facts(path, folder_name == MASK_FOLDER)
            facts["product"] = product_name
            facts["folder"] = folder_name
            facts["file_name"] = path.name
            inventory.append(facts)

integrity_lines = []
integrity_lines.append("Integrity report")
integrity_lines.append("Files read: " + str(len(inventory)))
print("Files read:", len(inventory))

## 8. Check 1: counts against the paper

The expected numbers are typed in from Table IV of the MVTec AD 2 paper (arXiv 2503.21622v1). Every defective test image should have one mask, so the mask count equals the defective count.

In [ ]:
EXPECTED_COUNTS = {
    "sheet_metal": {
        "train/good": 137,
        "validation/good": 19,
        "test_public/good": 24,
        "test_public/bad": 90,
        "test_public/ground_truth/bad": 90,
        "test_private": 142,
        "test_private_mixed": 142,
    },
    "vial": {
        "train/good": 291,
        "validation/good": 41,
        "test_public/good": 35,
        "test_public/bad": 105,
        "test_public/ground_truth/bad": 105,
        "test_private": 276,
        "test_private_mixed": 276,
    },
}

integrity_lines.append("")
integrity_lines.append("Check 1: counts against the paper")
count_problems = 0

for product_name in PRODUCT_NAMES:
    for folder_name in CHECK_FOLDERS:
        found = 0
        for row in inventory:
            if row["product"] == product_name and row["folder"] == folder_name:
                found = found + 1
        expected = EXPECTED_COUNTS[product_name][folder_name]
        result = "ok"
        if found != expected:
            result = "MISMATCH"
            count_problems = count_problems + 1
        line = product_name + " " + folder_name + ": found " + str(found)
        line = line + ", expected " + str(expected) + " " + result
        integrity_lines.append(line)

integrity_lines.append("Check 1 problems: " + str(count_problems))
print("Check 1 problems:", count_problems)

## 9. Checks 2 and 3: open every file, record size and channels

A damaged file could crash a long GPU run, so any file that failed to open is listed. Mode `L` means one greyscale channel, and `RGB` means three colour channels. The sizes are compared with Table IV of the paper.

In [ ]:
EXPECTED_SIZES = {"sheet_metal": (4224, 1056), "vial": (1400, 1900)}

integrity_lines.append("")
integrity_lines.append("Check 2: open every file")
open_problems = 0

for row in inventory:
    if not row["opened"]:
        open_problems = open_problems + 1
        integrity_lines.append("FAILED: " + row["product"] + " " + row["folder"] + " " + row["file_name"])

integrity_lines.append("Check 2 problems: " + str(open_problems))

integrity_lines.append("")
integrity_lines.append("Check 3: sizes and channels (number of files, then description)")
size_problems = 0
combination_counts = {}

for row in inventory:
    if row["opened"]:
        combination = (row["product"], row["width"], row["height"], row["mode"])
        if combination not in combination_counts:
            combination_counts[combination] = 0
        combination_counts[combination] = combination_counts[combination] + 1
        if (row["width"], row["height"]) != EXPECTED_SIZES[row["product"]]:
            size_problems = size_problems + 1
            integrity_lines.append("UNEXPECTED SIZE: " + row["product"] + " " + row["folder"] + " " + row["file_name"])

for combination in sorted(combination_counts):
    product_name, width, height, mode = combination
    line = str(combination_counts[combination]) + "  " + product_name
    line = line + " width " + str(width) + " height " + str(height) + " mode " + mode
    integrity_lines.append(line)

integrity_lines.append("Check 3 problems: " + str(size_problems))
print("Check 2 problems:", open_problems)
print("Check 3 problems:", size_problems)

## 10. Check 4: masks

Wrong masks would silently corrupt AU-PRO and pixel F1. Every defective image needs a mask named `<image name>_mask.png`, of the same size, holding only 0 and 255, with at least one defect pixel. Every mask needs an image, and good images should have no mask folder.

In [ ]:
integrity_lines.append("")
integrity_lines.append("Check 4: masks")
mask_problems = []

for product_name in PRODUCT_NAMES:
    # Dictionaries give a quick look-up of a file's facts by its name
    defective_images = {}
    masks = {}
    for row in inventory:
        if row["product"] == product_name and row["folder"] == "test_public/bad":
            defective_images[row["file_name"]] = row
        if row["product"] == product_name and row["folder"] == MASK_FOLDER:
            masks[row["file_name"]] = row

    for image_name in defective_images:
        image_row = defective_images[image_name]
        mask_name = image_name.replace(".png", "_mask.png")
        label = product_name + " " + mask_name

        if mask_name not in masks:
            mask_problems.append("NO MASK: " + label)
            continue

        mask_row = masks[mask_name]
        if not mask_row["opened"] or not image_row["opened"]:
            mask_problems.append("COULD NOT COMPARE, file did not open: " + label)
            continue

        if (mask_row["width"], mask_row["height"]) != (image_row["width"], image_row["height"]):
            mask_problems.append("WRONG SIZE: " + label)

        for value in mask_row["mask_values"]:
            if value != 0 and value != 255:
                mask_problems.append("UNEXPECTED VALUE " + str(value) + ": " + label)

        # A defective image whose mask has no defect pixels cannot be scored properly
        if 255 not in mask_row["mask_values"]:
            mask_problems.append("NO DEFECT PIXELS: " + label)

    for mask_name in masks:
        image_name = mask_name.replace("_mask.png", ".png")
        if image_name not in defective_images:
            mask_problems.append("MASK WITHOUT IMAGE: " + product_name + " " + mask_name)

    # Masks should only exist for defective images
    ground_truth_folder = product_roots[product_name] / product_name / "test_public" / "ground_truth"
    for path in ground_truth_folder.iterdir():
        if path.name != "bad":
            mask_problems.append("UNEXPECTED MASK FOLDER: " + product_name + " " + path.name)

for problem in mask_problems:
    integrity_lines.append(problem)

integrity_lines.append("Check 4 problems: " + str(len(mask_problems)))
print("Check 4 problems:", len(mask_problems))

## 11. Check 5: duplicate images

A training image repeated in the test set would make results look better than they are. Two files with the same fingerprint are byte-identical. Masks are left out here, because the same defect under different lighting can have the same mask; check 6 looks at that.

**Limit:** this finds byte-identical files only. It does not find the same scene under different lighting, or a re-saved copy.

In [ ]:
integrity_lines.append("")
integrity_lines.append("Check 5: duplicate images (byte-identical files)")

# Maps each fingerprint to every image that has it
files_by_fingerprint = {}

for row in inventory:
    if row["folder"] != MASK_FOLDER:
        fingerprint = row["fingerprint"]
        if fingerprint not in files_by_fingerprint:
            files_by_fingerprint[fingerprint] = []
        files_by_fingerprint[fingerprint].append(row)

duplicate_groups = 0

for fingerprint in files_by_fingerprint:
    group = files_by_fingerprint[fingerprint]
    if len(group) > 1:
        duplicate_groups = duplicate_groups + 1
        names = []
        for row in group:
            names.append(row["product"] + "/" + row["folder"] + "/" + row["file_name"])
        integrity_lines.append("DUPLICATE: " + ", ".join(names))

integrity_lines.append("Check 5 problems: " + str(duplicate_groups))
print("Check 5 problems:", duplicate_groups)

## 12. Check 6: scene pattern in the public test set

File names look like `000_regular.png`, a scene number and a lighting condition. This matters for splitting: images of one scene in both halves would make half B less independent. We check that each number has the same lighting conditions, and count how many defective scenes have byte-identical masks across their lighting conditions.

**Limit:** this comes from names and mask files. It does not prove the images show the same scene. Phase 2 checks that by looking at them.

In [ ]:
integrity_lines.append("")
integrity_lines.append("Check 6: scene pattern in the public test set")
scene_problems = 0

for product_name in PRODUCT_NAMES:
    for folder_name in ["test_public/good", "test_public/bad"]:
        conditions_by_scene = {}
        for row in inventory:
            if row["product"] == product_name and row["folder"] == folder_name:
                # 000_shift_1.png splits into the number 000 and the condition shift_1
                stem = row["file_name"].replace(".png", "")
                scene, condition = stem.split("_", 1)
                if scene not in conditions_by_scene:
                    conditions_by_scene[scene] = []
                conditions_by_scene[scene].append(condition)

        all_conditions = []
        for scene in sorted(conditions_by_scene):
            conditions = sorted(conditions_by_scene[scene])
            if conditions not in all_conditions:
                all_conditions.append(conditions)

        line = product_name + " " + folder_name + ": " + str(len(conditions_by_scene)) + " scene numbers"
        integrity_lines.append(line)
        for conditions in all_conditions:
            integrity_lines.append("  conditions: " + ", ".join(conditions))
        if len(all_conditions) > 1:
            scene_problems = scene_problems + 1
            integrity_lines.append("  DIFFERENT CONDITIONS between scene numbers")

    # Collect the different mask fingerprints for each defective scene number
    fingerprints_by_scene = {}
    for row in inventory:
        if row["product"] == product_name and row["folder"] == MASK_FOLDER:
            scene = row["file_name"].split("_", 1)[0]
            if scene not in fingerprints_by_scene:
                fingerprints_by_scene[scene] = []
            if row["fingerprint"] not in fingerprints_by_scene[scene]:
                fingerprints_by_scene[scene].append(row["fingerprint"])

    identical_scenes = 0
    for scene in fingerprints_by_scene:
        if len(fingerprints_by_scene[scene]) == 1:
            identical_scenes = identical_scenes + 1
    line = product_name + ": " + str(identical_scenes) + " of " + str(len(fingerprints_by_scene))
    integrity_lines.append(line + " defective scene numbers have one identical mask across all conditions")

integrity_lines.append("Check 6 problems: " + str(scene_problems))
print("Check 6 problems:", scene_problems)

## 13. Save the report and the inventory

`integrity_report.txt` is the short report. `image_inventory.csv` has one row per file, with its facts and fingerprint. It holds no image content, so it can go to GitHub.

In [ ]:
import csv

with open("integrity_report.txt", "w") as report_file:
    for line in integrity_lines:
        report_file.write(line + "\n")

CSV_COLUMNS = ["product", "folder", "file_name", "opened", "width", "height", "mode", "fingerprint"]

# newline="" stops the csv library adding blank lines between rows
with open("image_inventory.csv", "w", newline="") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(CSV_COLUMNS)
    for row in inventory:
        values = []
        for column in CSV_COLUMNS:
            values.append(row[column])
        writer.writerow(values)

for line in integrity_lines:
    print(line)